# Private RLHF: Training

This notebook contains the training and evaluation pipeline used for the paper
**“Privacy-Preserving Reinforcement Learning from Human Feedback via Decoupled Reward Modeling.”**

It trains the private reward models, the DP-DPO baseline, and the split DP policy-optimization baseline. The notebook is organized into small executable sections while preserving the original experimental code and settings.

## Install dependencies

In [ ]:
!pip -q install "transformers>=4.40.0" datasets peft opacus accelerate

import os, re, json, time, gc, inspect
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Optional

from google.colab import drive
from datasets import load_dataset
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    PreTrainedTokenizerBase,
)
from peft import LoraConfig, get_peft_model
from opacus import PrivacyEngine
from opacus.utils.batch_memory_manager import BatchMemoryManager
from tqdm.auto import tqdm

## Mount Google Drive

In [ ]:
drive.mount("/content/drive", force_remount=True)

## Project paths

All generated CSV files, caches, and model artifacts are stored under `ROOT`. Change only the `ROOT` line when using a different Drive folder.

In [ ]:
ROOT = "/content/drive/MyDrive/Private_Finetuning"
MASTER_CSV = os.path.join(ROOT, "master_results.csv")
ARTIFACTS_ROOT = os.path.join(ROOT, "artifacts")
CACHE_ROOT = os.path.join(ROOT, "cache")

os.makedirs(ROOT, exist_ok=True)
os.makedirs(ARTIFACTS_ROOT, exist_ok=True)
os.makedirs(CACHE_ROOT, exist_ok=True)

print("ROOT       =", ROOT)
print("MASTER_CSV =", MASTER_CSV)
print("ARTIFACTS  =", ARTIFACTS_ROOT)
print("CACHE      =", CACHE_ROOT)

## Experiment configuration

This cell contains the experimental settings used by the notebook.

In [ ]:
MODEL_ID = "google/gemma-2b-it"

EPSILONS = [0.5, 1.0, 2.0]
SEEDS    = [11, 22, 33]
EPOCHS   = 2

DELTA = 1e-5
MAX_GRAD_NORM = 1.0
POISSON = True

LR_RM  = 1e-3
LR_POL = 1e-4

BETA_DPO = 0.5
BETA_KL  = 0.5
EPS_CLIP = 0.2

# ✅ PPO rho stabilization
C_LOGRHO = 20.0  # chosen based on clamp_frac scan; exp overflow-proof

N_TOTAL   = 40000
TEST_FRAC = 0.2
MAX_LEN   = 256

LOGICAL_BATCH_SIZE  = 64
PHYSICAL_BATCH_SIZE = 8

EVAL_BATCH_SIZE_RM  = 32
EVAL_BATCH_SIZE_POL = 4

# caching batch sizes (no grad)
CACHE_BATCH_REF = 4
CACHE_BATCH_ADV = 16

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "GPU is required."
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print("DEVICE =", DEVICE)

## Gradient-clipping diagnostics

In [ ]:
CLIP_WINDOWS: Dict[str, Tuple[int, int]] = {
    "clip_warmup_0_49":      (0, 50),
    "clip_win_1000_1049":    (1000, 1050),
    "clip_win_2000_2049":    (2000, 2050),
    "clip_win_4000_4049":    (4000, 4050),
    "clip_win_7000_7049":    (7000, 7050),
}
def clip_window_hits(global_step: int) -> List[str]:
    return [k for k, (a, b) in CLIP_WINDOWS.items() if a <= global_step < b]

@torch.no_grad()
def per_sample_norms_sq_from_grad_sample(model: torch.nn.Module) -> torch.Tensor:
    norms_sq = None
    for _, p in model.named_parameters():
        if not p.requires_grad:
            continue
        gs = getattr(p, "grad_sample", None)
        if gs is None or (not torch.is_tensor(gs)):
            continue
        g2 = (gs.float().reshape(gs.shape[0], -1) ** 2).sum(dim=1)
        norms_sq = g2 if norms_sq is None else (norms_sq + g2)
    if norms_sq is None:
        return torch.zeros(0, device=next(model.parameters()).device)
    return norms_sq

def cleanup(*objs):
    for o in objs:
        try:
            del o
        except:
            pass
    gc.collect()
    torch.cuda.empty_cache()

def unwrap_model(m: torch.nn.Module) -> torch.nn.Module:
    return getattr(m, "_module", getattr(m, "module", m))

def safe_get_epsilon(pe: PrivacyEngine, delta: float) -> float:
    try:
        return float(pe.get_epsilon(delta=delta))
    except Exception:
        return float("nan")

## Opacus compatibility helpers

In [ ]:
def make_privacy_engine_rdp() -> PrivacyEngine:
    sig = inspect.signature(PrivacyEngine)
    if "accountant" in sig.parameters:
        try:
            return PrivacyEngine(accountant="rdp")
        except Exception:
            return PrivacyEngine()
    return PrivacyEngine()

def make_private_with_epsilon_safe(pe: PrivacyEngine, **kwargs):
    sig = inspect.signature(pe.make_private_with_epsilon)
    allowed = set(sig.parameters.keys())
    filtered = {k: v for k, v in kwargs.items() if k in allowed}
    return pe.make_private_with_epsilon(**filtered)

def count_trainable_params(m: torch.nn.Module) -> int:
    return int(sum(p.numel() for p in m.parameters() if p.requires_grad))

## Experiment log utilities

`master_results.csv` is maintained as the experiment ledger. Completed runs are detected from this file so interrupted Colab sessions can resume.

In [ ]:
BASE_COLS = [
    "timestamp", "run_id", "method",
    "epsilon_target", "epsilon_spent", "delta",
    "seed", "epochs", "lr",
    "max_grad_norm", "poisson_sampling",
    "n_total", "test_frac", "max_len",
    "logical_batch", "physical_batch",
    "metric_name", "metric_value",
    "train_minutes", "eval_minutes",
    "train_status", "eval_status",
    "artifact_dir",
    "beta", "beta_kl", "eps_clip",
    "lora_r", "lora_alpha", "lora_dropout", "lora_last_k",
    "split_role", "rm_run_id", "rm_artifact_dir",
    "kl_penalty_name",
] + list(CLIP_WINDOWS.keys())

def _atomic_to_csv(df: pd.DataFrame, path: str):
    tmp = path + f".tmp_{int(time.time())}"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def load_master_df(path: str) -> pd.DataFrame:
    if os.path.exists(path):
        df = pd.read_csv(path)
        for c in BASE_COLS:
            if c not in df.columns:
                df[c] = np.nan
        return df
    return pd.DataFrame(columns=BASE_COLS)

def append_row(path: str, row: dict):
    df = load_master_df(path)
    for k in row.keys():
        if k not in df.columns:
            df[k] = np.nan
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    _atomic_to_csv(df, path)

def update_row(path: str, run_id: str, updates: dict):
    df = load_master_df(path)
    mask = (df["run_id"].astype(str) == str(run_id))
    if mask.any():
        idx = df.index[mask][0]
        for k, v in updates.items():
            if k not in df.columns:
                df[k] = np.nan
            df.at[idx, k] = v
        _atomic_to_csv(df, path)
    else:
        append_row(path, {"run_id": run_id, **updates})

def is_done(
    df: pd.DataFrame,
    *,
    method: str,
    eps: float,
    seed: int,
    lr: float,
    epochs: int,
    max_grad_norm: float,
    beta: Optional[float]=None,
    beta_kl: Optional[float]=None,
    eps_clip: Optional[float]=None,
    split_role: Optional[str]=None,
) -> bool:
    if df.empty:
        return False

    sub = df[
        (df["method"] == method) &
        (pd.to_numeric(df["epsilon_target"], errors="coerce") == float(eps)) &
        (pd.to_numeric(df["seed"], errors="coerce") == int(seed)) &
        (pd.to_numeric(df["lr"], errors="coerce") == float(lr)) &
        (pd.to_numeric(df["epochs"], errors="coerce") == int(epochs)) &
        (pd.to_numeric(df["max_grad_norm"], errors="coerce") == float(max_grad_norm))
    ]
    if split_role is not None:
        sub = sub[sub["split_role"].astype(str) == str(split_role)]
    if beta is not None:
        sub = sub[pd.to_numeric(sub["beta"], errors="coerce") == float(beta)]
    if beta_kl is not None:
        sub = sub[pd.to_numeric(sub["beta_kl"], errors="coerce") == float(beta_kl)]
    if eps_clip is not None:
        sub = sub[pd.to_numeric(sub["eps_clip"], errors="coerce") == float(eps_clip)]
    if sub.empty:
        return False

    ok_train = sub["train_status"].astype(str).eq("ok")
    ok_eval  = sub["eval_status"].astype(str).eq("ok")
    mv = pd.to_numeric(sub["metric_value"], errors="coerce")
    return bool((ok_train & ok_eval & mv.notna()).any())

## Tokenizer and pairwise preprocessing

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

PROMPT_PAT = re.compile(r"\n\nAssistant:\s*")

def extract_prompt_prefix(text: str) -> str:
    ms = list(PROMPT_PAT.finditer(text))
    if not ms:
        return ""
    return text[: ms[-1].end()]

def preprocess_pairwise_policy(examples):
    out = {
        "chosen_input_ids": [], "chosen_attention_mask": [],
        "rejected_input_ids": [], "rejected_attention_mask": [],
        "prompt_len": [],
    }
    for chosen, rejected in zip(examples["chosen"], examples["rejected"]):
        prompt = extract_prompt_prefix(chosen)
        t_prompt = tokenizer(prompt, truncation=True, max_length=MAX_LEN)
        prompt_len = len(t_prompt["input_ids"])

        tc = tokenizer(chosen, truncation=True, max_length=MAX_LEN)
        tr = tokenizer(rejected, truncation=True, max_length=MAX_LEN)

        max_prompt = min(len(tc["input_ids"]), len(tr["input_ids"])) - 1
        prompt_len = max(1, min(prompt_len, max_prompt))

        out["chosen_input_ids"].append(tc["input_ids"])
        out["chosen_attention_mask"].append(tc["attention_mask"])
        out["rejected_input_ids"].append(tr["input_ids"])
        out["rejected_attention_mask"].append(tr["attention_mask"])
        out["prompt_len"].append(prompt_len)
    return out

@dataclass
class PairCollator:
    tokenizer: PreTrainedTokenizerBase
    def __call__(self, feats: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        ch = [{"input_ids": f["chosen_input_ids"], "attention_mask": f["chosen_attention_mask"]} for f in feats]
        rj = [{"input_ids": f["rejected_input_ids"], "attention_mask": f["rejected_attention_mask"]} for f in feats]
        chb = self.tokenizer.pad(ch, padding=True, return_tensors="pt")
        rjb = self.tokenizer.pad(rj, padding=True, return_tensors="pt")
        prompt_lens = torch.tensor([f["prompt_len"] for f in feats], dtype=torch.long)

        batch = dict(
            chosen_input_ids=chb["input_ids"],
            chosen_attention_mask=chb["attention_mask"],
            rejected_input_ids=rjb["input_ids"],
            rejected_attention_mask=rjb["attention_mask"],
            prompt_len=prompt_lens,
        )

        # pass-through cached scalars if present
        for k in ["ref_delta", "ref_sum_c", "ref_sum_r", "ref_cnt_c", "ref_cnt_r", "adv"]:
            if k in feats[0]:
                batch[k] = torch.tensor([float(f[k]) for f in feats], dtype=torch.float32)

        return batch

collator = PairCollator(tokenizer=tokenizer)

## Response-only log-likelihood helpers

In [ ]:
def seq_logp_response_only_sum_and_count(model, input_ids, attention_mask, prompt_len, pad_id):
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits
    labels = input_ids[:, 1:].contiguous()
    logits = logits[:, :-1, :].contiguous()
    attn   = attention_mask[:, 1:].contiguous()

    B, Tm1 = labels.shape
    idx = torch.arange(Tm1, device=labels.device).view(1, -1).expand(B, -1)

    resp_mask = (idx >= (prompt_len.view(-1, 1) - 1)).to(attn.dtype)
    not_pad   = (labels != pad_id).to(attn.dtype)
    mask = attn.to(attn.dtype) * resp_mask * not_pad

    logp_tok = torch.gather(F.log_softmax(logits.float(), dim=-1), dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)
    sum_logp = (logp_tok * mask).sum(dim=1)
    n_resp = mask.sum(dim=1).clamp(min=1.0)
    return sum_logp, n_resp

def seq_logp_response_only_sum(model, input_ids, attention_mask, prompt_len, pad_id):
    s, _ = seq_logp_response_only_sum_and_count(model, input_ids, attention_mask, prompt_len, pad_id)
    return s

## Last-layer LoRA policy

In [ ]:
def _extract_layer_index_from_name(name: str) -> Optional[int]:
    for pat in [r"\.layers\.(\d+)\.", r"\.h\.(\d+)\."]:
        m = re.search(pat, name)
        if m:
            return int(m.group(1))
    return None

def restrict_lora_to_last_k_layers_strict(peft_model: torch.nn.Module, k: int = 1):
    lora_param_names = [n for (n, _) in peft_model.named_parameters() if "lora_" in n]
    idxs = []
    for n in lora_param_names:
        li = _extract_layer_index_from_name(n)
        if li is not None:
            idxs.append(li)
    if len(idxs) == 0:
        raise RuntimeError(
            "Cannot parse layer indices from LoRA parameter names; cannot enforce last-layer LoRA.\n"
            "Sample names:\n" + "\n".join(lora_param_names[:20])
        )

    max_idx = max(idxs)
    keep = set(range(max_idx - k + 1, max_idx + 1))

    for n, p in peft_model.named_parameters():
        if "lora_" in n:
            p.requires_grad_(False)

    kept_layers = set()
    kept_cnt = 0
    for n, p in peft_model.named_parameters():
        if "lora_" not in n:
            continue
        li = _extract_layer_index_from_name(n)
        if li is None:
            raise RuntimeError(f"LoRA param missing layer index pattern: {n}")
        if li in keep:
            p.requires_grad_(True)
            kept_layers.add(li)
            kept_cnt += 1

    if kept_layers != keep:
        raise RuntimeError(f"LoRA restriction mismatch. expected={sorted(list(keep))}, got={sorted(list(kept_layers))}")

    print(f"[LoRA STRICT] max_layer_idx={max_idx} | keep_layers={sorted(list(keep))} | kept_lora_params={kept_cnt}")
    return {"max_layer_idx": max_idx, "kept_layers": sorted(list(keep)), "kept_lora_params": kept_cnt}

def make_policy_last_layer_lora(model_id: str, *, r=8, alpha=32, dropout=0.0, last_k=1):
    base = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).to(DEVICE)
    base.config.pad_token_id = tokenizer.pad_token_id

    lora_cfg = LoraConfig(
        r=r, lora_alpha=alpha, lora_dropout=dropout,
        bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    policy = get_peft_model(base, lora_cfg).to(DEVICE).float()
    policy.train()

    _ = restrict_lora_to_last_k_layers_strict(policy, k=last_k)
    print(f"[Trainable params] policy={count_trainable_params(policy):,}")
    return policy

## Reward-model head utilities

In [ ]:
def make_reward_model_head_only(model_id: str) -> Tuple[torch.nn.Module, str]:
    rm = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=1).to(DEVICE).float()
    rm.config.pad_token_id = tokenizer.pad_token_id

    for p in rm.parameters():
        p.requires_grad_(False)

    head_attr = None
    for cand in ["score", "classifier"]:
        if hasattr(rm, cand):
            head_attr = cand
            break
    if head_attr is None:
        raise ValueError("Cannot find RM head attribute (expected .score or .classifier).")

    for p in getattr(rm, head_attr).parameters():
        p.requires_grad_(True)

    rm.train()
    print(f"[Trainable params] rm_head_only={count_trainable_params(rm):,} (head_attr={head_attr})")
    return rm, head_attr

def save_rm_head(rm: torch.nn.Module, head_attr: str, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    rm_inner = unwrap_model(rm)
    head = getattr(rm_inner, head_attr)
    torch.save(head.state_dict(), os.path.join(out_dir, "rm_head.pt"))
    meta = {"head_attr": head_attr, "model_id": MODEL_ID, "max_len": MAX_LEN}
    with open(os.path.join(out_dir, "rm_head_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    tokenizer.save_pretrained(out_dir)

def load_rm_head(out_dir: str, *, dtype=torch.float16) -> torch.nn.Module:
    with open(os.path.join(out_dir, "rm_head_meta.json"), "r") as f:
        meta = json.load(f)
    head_attr = meta["head_attr"]

    rm = AutoModelForSequenceClassification.from_pretrained(
        meta["model_id"], num_labels=1, torch_dtype=dtype
    ).to(DEVICE)
    rm.config.pad_token_id = tokenizer.pad_token_id
    for p in rm.parameters():
        p.requires_grad_(False)

    head = getattr(rm, head_attr)
    sd = torch.load(os.path.join(out_dir, "rm_head.pt"), map_location="cpu")
    head.load_state_dict(sd)
    for p in head.parameters():
        p.requires_grad_(False)

    rm.eval()
    return rm

## Evaluation metrics

In [ ]:
@torch.no_grad()
def eval_reward_accuracy_rm(rm: torch.nn.Module, test_dl: DataLoader) -> float:
    rm.eval()
    correct, total = 0, 0
    for batch in tqdm(test_dl, desc="eval_rm", leave=False):
        ch_ids = batch["chosen_input_ids"].to(DEVICE)
        ch_am  = batch["chosen_attention_mask"].to(DEVICE)
        rj_ids = batch["rejected_input_ids"].to(DEVICE)
        rj_am  = batch["rejected_attention_mask"].to(DEVICE)
        rc = rm(input_ids=ch_ids, attention_mask=ch_am).logits.squeeze(-1)
        rr = rm(input_ids=rj_ids, attention_mask=rj_am).logits.squeeze(-1)
        correct += (rc > rr).sum().item()
        total += rc.numel()
    return correct / max(total, 1)

@torch.no_grad()
def eval_win_rate_policy(policy: torch.nn.Module, test_dl: DataLoader) -> float:
    policy.eval()
    correct, total = 0, 0
    for batch in tqdm(test_dl, desc="eval_policy", leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items() if torch.is_tensor(v)}
        pol_c = seq_logp_response_only_sum(policy, batch["chosen_input_ids"], batch["chosen_attention_mask"],
                                           batch["prompt_len"], tokenizer.pad_token_id)
        pol_r = seq_logp_response_only_sum(policy, batch["rejected_input_ids"], batch["rejected_attention_mask"],
                                           batch["prompt_len"], tokenizer.pad_token_id)
        correct += (pol_c > pol_r).sum().item()
        total += pol_c.numel()
    return correct / max(total, 1)

## Reference-policy cache

Reference-policy quantities are computed once per seed and stored in Drive.

In [ ]:
def ref_cache_path(seed: int, n: int) -> str:
    return os.path.join(CACHE_ROOT, f"refcache_train_seed{seed}_n{n}_maxlen{MAX_LEN}.npz")

def ensure_reference_cache_train(train_tok, seed: int):
    needed = ["ref_delta", "ref_sum_c", "ref_sum_r", "ref_cnt_c", "ref_cnt_r"]
    if all(k in train_tok.column_names for k in needed):
        return train_tok

    path = ref_cache_path(seed, len(train_tok))
    if os.path.exists(path):
        z = np.load(path)
        ref_delta = z["ref_delta"].astype(np.float32)
        ref_sum_c = z["ref_sum_c"].astype(np.float32)
        ref_sum_r = z["ref_sum_r"].astype(np.float32)
        ref_cnt_c = z["ref_cnt_c"].astype(np.float32)
        ref_cnt_r = z["ref_cnt_r"].astype(np.float32)
        print(f"[REF CACHE] loaded: {path}")
    else:
        print(f"[REF CACHE] computing (seed={seed}) ...")
        reference = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
        reference.config.pad_token_id = tokenizer.pad_token_id
        reference.eval()
        for p in reference.parameters():
            p.requires_grad_(False)

        dl = DataLoader(train_tok, batch_size=CACHE_BATCH_REF, shuffle=False, collate_fn=collator, drop_last=False)

        sum_c_list, sum_r_list, cnt_c_list, cnt_r_list = [], [], [], []
        with torch.inference_mode():
            for batch in tqdm(dl, desc=f"ref_cache(seed={seed})", leave=False):
                batch = {k: v.to(DEVICE) for k, v in batch.items() if torch.is_tensor(v)}
                sc, nc = seq_logp_response_only_sum_and_count(reference, batch["chosen_input_ids"], batch["chosen_attention_mask"],
                                                              batch["prompt_len"], tokenizer.pad_token_id)
                sr, nr = seq_logp_response_only_sum_and_count(reference, batch["rejected_input_ids"], batch["rejected_attention_mask"],
                                                              batch["prompt_len"], tokenizer.pad_token_id)
                sum_c_list.append(sc.detach().cpu().numpy())
                sum_r_list.append(sr.detach().cpu().numpy())
                cnt_c_list.append(nc.detach().cpu().numpy())
                cnt_r_list.append(nr.detach().cpu().numpy())

        ref_sum_c = np.concatenate(sum_c_list, axis=0).astype(np.float32)
        ref_sum_r = np.concatenate(sum_r_list, axis=0).astype(np.float32)
        ref_cnt_c = np.concatenate(cnt_c_list, axis=0).astype(np.float32)
        ref_cnt_r = np.concatenate(cnt_r_list, axis=0).astype(np.float32)
        ref_delta = (ref_sum_c - ref_sum_r).astype(np.float32)

        np.savez_compressed(path,
                            ref_delta=ref_delta,
                            ref_sum_c=ref_sum_c, ref_sum_r=ref_sum_r,
                            ref_cnt_c=ref_cnt_c, ref_cnt_r=ref_cnt_r)
        print(f"[REF CACHE] saved: {path}")
        cleanup(reference, dl)

    train_tok = train_tok.add_column("ref_delta", ref_delta.tolist())
    train_tok = train_tok.add_column("ref_sum_c", ref_sum_c.tolist())
    train_tok = train_tok.add_column("ref_sum_r", ref_sum_r.tolist())
    train_tok = train_tok.add_column("ref_cnt_c", ref_cnt_c.tolist())
    train_tok = train_tok.add_column("ref_cnt_r", ref_cnt_r.tolist())
    return train_tok

## Reward-margin cache

Reward margins for the policy-optimization split are cached by privacy level, seed, and reward-model run.

In [ ]:
def adv_cache_path(seed: int, eps: float, rm_run_id: str, n: int) -> str:
    safe = re.sub(r"[^A-Za-z0-9_\-\.]", "_", rm_run_id)
    return os.path.join(CACHE_ROOT, f"advcache_po_seed{seed}_eps{eps}_n{n}_rmrun{safe}.npz")

def ensure_adv_cache_po(po_half, seed: int, eps: float, rm_run_id: str, rm_artifact_dir: str):
    if "adv" in po_half.column_names:
        return po_half

    path = adv_cache_path(seed, eps, rm_run_id, len(po_half))
    if os.path.exists(path):
        z = np.load(path)
        adv = z["adv"].astype(np.float32)
        print(f"[ADV CACHE] loaded: {path}")
    else:
        print(f"[ADV CACHE] computing (seed={seed}, eps={eps}) ...")
        rm = load_rm_head(rm_artifact_dir, dtype=torch.float16).to(DEVICE)
        rm.eval()

        dl = DataLoader(po_half, batch_size=CACHE_BATCH_ADV, shuffle=False, collate_fn=collator, drop_last=False)
        adv_list = []
        with torch.inference_mode():
            for batch in tqdm(dl, desc=f"adv_cache(seed={seed},eps={eps})", leave=False):
                batch = {k: v.to(DEVICE) for k, v in batch.items() if torch.is_tensor(v)}
                rc = rm(input_ids=batch["chosen_input_ids"], attention_mask=batch["chosen_attention_mask"]).logits.squeeze(-1)
                rr = rm(input_ids=batch["rejected_input_ids"], attention_mask=batch["rejected_attention_mask"]).logits.squeeze(-1)
                adv_list.append((rc - rr).detach().cpu().numpy())

        adv = np.concatenate(adv_list, axis=0).astype(np.float32)
        np.savez_compressed(path, adv=adv)
        print(f"[ADV CACHE] saved: {path}")
        cleanup(rm, dl)

    po_half = po_half.add_column("adv", adv.tolist())
    return po_half

## DP reward-model training

In [ ]:
def run_dp_rm_headonly(eps: float, seed: int, train_ds, test_ds, *, method: str, split_role: str):
    df = load_master_df(MASTER_CSV)
    if is_done(df, method=method, eps=eps, seed=seed, lr=LR_RM, epochs=EPOCHS, max_grad_norm=MAX_GRAD_NORM, split_role=split_role):
        print(f"[SKIP] {method}({split_role}) eps={eps} seed={seed}")
        return

    torch.manual_seed(seed); np.random.seed(seed)

    run_id = f"{method}_eps{eps}_seed{seed}_{int(time.time())}"
    artifact_dir = os.path.join(ARTIFACTS_ROOT, "rm", method, f"eps{eps}", f"seed{seed}", f"run_{run_id}")
    os.makedirs(artifact_dir, exist_ok=True)

    train_dl = DataLoader(train_ds, batch_size=LOGICAL_BATCH_SIZE, shuffle=True, collate_fn=collator, drop_last=False)
    test_dl  = DataLoader(test_ds,  batch_size=EVAL_BATCH_SIZE_RM, shuffle=False, collate_fn=collator, drop_last=False)

    rm, head_attr = make_reward_model_head_only(MODEL_ID)
    opt = torch.optim.AdamW([p for p in rm.parameters() if p.requires_grad], lr=LR_RM)

    pe = make_privacy_engine_rdp()
    rm, opt, train_dl_priv = make_private_with_epsilon_safe(
        pe,
        module=rm,
        optimizer=opt,
        data_loader=train_dl,
        epochs=EPOCHS,
        target_epsilon=eps,
        target_delta=DELTA,
        max_grad_norm=MAX_GRAD_NORM,
        poisson_sampling=POISSON,
    )

    clip_counts = {k: {"clipped": 0, "total": 0} for k in CLIP_WINDOWS.keys()}
    global_step = 0
    t0 = time.time()
    train_status = "ok"

    try:
        with BatchMemoryManager(
            data_loader=train_dl_priv,
            max_physical_batch_size=PHYSICAL_BATCH_SIZE,
            optimizer=opt
        ) as mem_dl:
            for ep in range(EPOCHS):
                pbar = tqdm(mem_dl, desc=f"[{method}|{split_role}] train {ep+1}/{EPOCHS}", leave=False)
                for batch in pbar:
                    batch = {k: v.to(DEVICE) for k, v in batch.items() if torch.is_tensor(v)}
                    ch = rm(input_ids=batch["chosen_input_ids"], attention_mask=batch["chosen_attention_mask"]).logits.squeeze(-1)
                    rj = rm(input_ids=batch["rejected_input_ids"], attention_mask=batch["rejected_attention_mask"]).logits.squeeze(-1)
                    loss = -F.logsigmoid(ch - rj).mean()

                    opt.zero_grad(set_to_none=True)
                    loss.backward()

                    hits = clip_window_hits(global_step)
                    if hits:
                        norms_sq = per_sample_norms_sq_from_grad_sample(rm)
                        if norms_sq.numel() > 0:
                            norms = torch.sqrt(norms_sq)
                            n = int(norms.numel())
                            n_clipped = int((norms > MAX_GRAD_NORM).sum().item())
                            for w in hits:
                                clip_counts[w]["clipped"] += n_clipped
                                clip_counts[w]["total"] += n

                    opt.step()
                    global_step += 1
                    pbar.set_postfix(loss=float(loss.detach().cpu()), gstep=global_step)

    except Exception as e:
        train_status = f"train_error:{type(e).__name__}"
        print(f"[TRAIN ERROR] {method}({split_role}) | {train_status} | {e}")

    train_minutes = (time.time() - t0) / 60.0
    eps_spent = safe_get_epsilon(pe, DELTA)

    if train_status == "ok":
        save_rm_head(rm, head_attr, artifact_dir)

    clip_fracs = {k: (clip_counts[k]["clipped"]/clip_counts[k]["total"] if clip_counts[k]["total"]>0 else np.nan)
                  for k in CLIP_WINDOWS.keys()}

    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    row = dict(
        timestamp=ts, run_id=run_id, method=method,
        epsilon_target=float(eps), epsilon_spent=float(eps_spent), delta=float(DELTA),
        seed=int(seed), epochs=int(EPOCHS), lr=float(LR_RM),
        max_grad_norm=float(MAX_GRAD_NORM), poisson_sampling=bool(POISSON),
        n_total=int(N_TOTAL), test_frac=float(TEST_FRAC), max_len=int(MAX_LEN),
        logical_batch=int(LOGICAL_BATCH_SIZE), physical_batch=int(PHYSICAL_BATCH_SIZE),
        metric_name="reward_accuracy", metric_value=np.nan,
        train_minutes=float(train_minutes), eval_minutes=np.nan,
        train_status=train_status, eval_status="train_only",
        artifact_dir=artifact_dir,
        beta=np.nan, beta_kl=np.nan, eps_clip=np.nan,
        lora_r=np.nan, lora_alpha=np.nan, lora_dropout=np.nan, lora_last_k=np.nan,
        split_role=str(split_role), rm_run_id=np.nan, rm_artifact_dir=np.nan,
        kl_penalty_name=np.nan,
        **clip_fracs
    )
    append_row(MASTER_CSV, row)

    if train_status != "ok":
        update_row(MASTER_CSV, run_id, dict(
            eval_status="skipped_train_error",
            metric_value=np.nan,
            eval_minutes=np.nan,
            epsilon_spent=float(eps_spent),
        ))
        cleanup(rm, opt, pe, train_dl, test_dl, train_dl_priv)
        return

    eval_status = "ok"
    acc = np.nan
    t1 = time.time()
    try:
        acc = eval_reward_accuracy_rm(load_rm_head(artifact_dir, dtype=torch.float16), test_dl)
    except Exception as e:
        eval_status = f"eval_error:{type(e).__name__}"
        print(f"[EVAL ERROR] {method}({split_role}) | {eval_status} | {e}")
    eval_minutes = (time.time() - t1) / 60.0

    update_row(MASTER_CSV, run_id, dict(
        metric_value=float(acc) if acc==acc else np.nan,
        eval_minutes=float(eval_minutes),
        eval_status=eval_status,
        epsilon_spent=float(safe_get_epsilon(pe, DELTA)),
        train_status="ok",
    ))

    print(f"[DONE] {method}({split_role}) eps={eps} seed={seed} | acc={(acc*100 if acc==acc else float('nan')):.2f}% | eps_spent={safe_get_epsilon(pe, DELTA):.3f}")
    print("  artifact_dir=", artifact_dir)
    cleanup(rm, opt, pe, train_dl, test_dl, train_dl_priv)

## DP-DPO training

In [ ]:
def run_dp_dpo_lastlora_cachedref(eps: float, seed: int, train_ds, test_ds):
    method = "dp_dpo"
    df = load_master_df(MASTER_CSV)
    if is_done(df, method=method, eps=eps, seed=seed, lr=LR_POL, epochs=EPOCHS, max_grad_norm=MAX_GRAD_NORM, beta=BETA_DPO):
        print(f"[SKIP] {method} eps={eps} seed={seed}")
        return

    assert "ref_delta" in train_ds.column_names, "train_ds must include ref_delta (ensure_reference_cache_train)."

    torch.manual_seed(seed); np.random.seed(seed)

    run_id = f"{method}_eps{eps}_beta{BETA_DPO}_seed{seed}_{int(time.time())}"
    artifact_dir = os.path.join(ARTIFACTS_ROOT, "dpo", method, f"eps{eps}", f"beta{BETA_DPO}", f"seed{seed}", f"run_{run_id}")
    os.makedirs(artifact_dir, exist_ok=True)

    train_dl = DataLoader(train_ds, batch_size=LOGICAL_BATCH_SIZE, shuffle=True, collate_fn=collator, drop_last=False)
    test_dl  = DataLoader(test_ds,  batch_size=EVAL_BATCH_SIZE_POL, shuffle=False, collate_fn=collator, drop_last=False)

    policy = make_policy_last_layer_lora(MODEL_ID, r=8, alpha=32, dropout=0.0, last_k=1)
    opt = torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad], lr=LR_POL)

    pe = make_privacy_engine_rdp()
    policy, opt, train_dl_priv = make_private_with_epsilon_safe(
        pe,
        module=policy,
        optimizer=opt,
        data_loader=train_dl,
        epochs=EPOCHS,
        target_epsilon=eps,
        target_delta=DELTA,
        max_grad_norm=MAX_GRAD_NORM,
        poisson_sampling=POISSON,
    )

    clip_counts = {k: {"clipped": 0, "total": 0} for k in CLIP_WINDOWS.keys()}
    global_step = 0
    t0 = time.time()
    train_status = "ok"

    try:
        with BatchMemoryManager(
            data_loader=train_dl_priv,
            max_physical_batch_size=PHYSICAL_BATCH_SIZE,
            optimizer=opt
        ) as mem_dl:
            for ep in range(EPOCHS):
                pbar = tqdm(mem_dl, desc=f"[dp_dpo cachedref] train {ep+1}/{EPOCHS}", leave=False)
                for batch in pbar:
                    batch = {k: v.to(DEVICE) for k, v in batch.items() if torch.is_tensor(v)}
                    opt.zero_grad(set_to_none=True)

                    pol_c = seq_logp_response_only_sum(policy, batch["chosen_input_ids"], batch["chosen_attention_mask"],
                                                       batch["prompt_len"], tokenizer.pad_token_id)
                    pol_r = seq_logp_response_only_sum(policy, batch["rejected_input_ids"], batch["rejected_attention_mask"],
                                                       batch["prompt_len"], tokenizer.pad_token_id)

                    ref_delta = batch["ref_delta"]
                    logits = (pol_c - pol_r) - ref_delta
                    loss = (-F.logsigmoid(BETA_DPO * logits)).mean()

                    if not torch.isfinite(loss):
                        raise RuntimeError("Non-finite loss detected in DPO.")

                    loss.backward()

                    hits = clip_window_hits(global_step)
                    if hits:
                        norms_sq = per_sample_norms_sq_from_grad_sample(policy)
                        if norms_sq.numel() > 0:
                            norms = torch.sqrt(norms_sq)
                            n = int(norms.numel())
                            n_clipped = int((norms > MAX_GRAD_NORM).sum().item())
                            for w in hits:
                                clip_counts[w]["clipped"] += n_clipped
                                clip_counts[w]["total"] += n

                    opt.step()
                    global_step += 1
                    pbar.set_postfix(loss=float(loss.detach().cpu()), gstep=global_step)

    except Exception as e:
        train_status = f"train_error:{type(e).__name__}"
        print(f"[TRAIN ERROR] {method} | {train_status} | {e}")

    train_minutes = (time.time() - t0) / 60.0
    eps_spent = safe_get_epsilon(pe, DELTA)

    clip_fracs = {k: (clip_counts[k]["clipped"]/clip_counts[k]["total"] if clip_counts[k]["total"]>0 else np.nan)
                  for k in CLIP_WINDOWS.keys()}

    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    row = dict(
        timestamp=ts, run_id=run_id, method=method,
        epsilon_target=float(eps), epsilon_spent=float(eps_spent), delta=float(DELTA),
        seed=int(seed), epochs=int(EPOCHS), lr=float(LR_POL),
        max_grad_norm=float(MAX_GRAD_NORM), poisson_sampling=bool(POISSON),
        n_total=int(N_TOTAL), test_frac=float(TEST_FRAC), max_len=int(MAX_LEN),
        logical_batch=int(LOGICAL_BATCH_SIZE), physical_batch=int(PHYSICAL_BATCH_SIZE),
        metric_name="win_rate", metric_value=np.nan,
        train_minutes=float(train_minutes), eval_minutes=np.nan,
        train_status=train_status, eval_status="train_only",
        artifact_dir=artifact_dir,
        beta=float(BETA_DPO), beta_kl=np.nan, eps_clip=np.nan,
        lora_r=8, lora_alpha=32, lora_dropout=0.0, lora_last_k=1,
        split_role="full", rm_run_id=np.nan, rm_artifact_dir=np.nan,
        kl_penalty_name=np.nan,
        **clip_fracs
    )
    append_row(MASTER_CSV, row)

    if train_status != "ok":
        update_row(MASTER_CSV, run_id, dict(
            eval_status="skipped_train_error",
            metric_value=np.nan,
            eval_minutes=np.nan,
            epsilon_spent=float(eps_spent),
        ))
        cleanup(policy, opt, pe, train_dl, test_dl, train_dl_priv)
        return

    unwrap_model(policy).save_pretrained(os.path.join(artifact_dir, "adapter"))
    tokenizer.save_pretrained(os.path.join(artifact_dir, "adapter"))

    eval_status = "ok"
    win = np.nan
    t1 = time.time()
    try:
        win = eval_win_rate_policy(policy, test_dl)
    except Exception as e:
        eval_status = f"eval_error:{type(e).__name__}"
        print(f"[EVAL ERROR] {method} | {eval_status} | {e}")
    eval_minutes = (time.time() - t1) / 60.0

    update_row(MASTER_CSV, run_id, dict(
        metric_value=float(win) if win==win else np.nan,
        eval_minutes=float(eval_minutes),
        eval_status=eval_status,
        epsilon_spent=float(safe_get_epsilon(pe, DELTA)),
        train_status="ok",
    ))

    print(f"[DONE] dp_dpo(cachedref) eps={eps} seed={seed} | win={(win*100 if win==win else float('nan')):.2f}% | eps_spent={safe_get_epsilon(pe, DELTA):.3f}")
    print("  artifact_dir=", artifact_dir)
    cleanup(policy, opt, pe, train_dl, test_dl, train_dl_priv)

## Split DP policy-optimization baseline

In [ ]:
def run_dp_ppo_like_split_cached(eps: float, seed: int, po_train_ds, test_ds, rm_run_id: str, rm_artifact_dir: str):
    method = "dp_ppo_like_split"
    df = load_master_df(MASTER_CSV)
    if is_done(df, method=method, eps=eps, seed=seed, lr=LR_POL, epochs=EPOCHS, max_grad_norm=MAX_GRAD_NORM,
               beta_kl=BETA_KL, eps_clip=EPS_CLIP, split_role="po_half"):
        print(f"[SKIP] {method} eps={eps} seed={seed}")
        return

    for k in ["ref_delta", "ref_sum_c", "ref_sum_r", "adv"]:
        assert k in po_train_ds.column_names, f"po_train_ds must include {k} (ref cache + adv cache)."

    torch.manual_seed(seed); np.random.seed(seed)

    run_id = f"{method}_eps{eps}_betaKL{BETA_KL}_seed{seed}_{int(time.time())}"
    artifact_dir = os.path.join(ARTIFACTS_ROOT, "ppo_like", method, f"eps{eps}", f"betaKL{BETA_KL}", f"seed{seed}", f"run_{run_id}")
    os.makedirs(artifact_dir, exist_ok=True)

    train_dl = DataLoader(po_train_ds, batch_size=LOGICAL_BATCH_SIZE, shuffle=True, collate_fn=collator, drop_last=False)
    test_dl  = DataLoader(test_ds,     batch_size=EVAL_BATCH_SIZE_POL, shuffle=False, collate_fn=collator, drop_last=False)

    policy = make_policy_last_layer_lora(MODEL_ID, r=8, alpha=32, dropout=0.0, last_k=1)
    opt = torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad], lr=LR_POL)

    pe = make_privacy_engine_rdp()
    policy, opt, train_dl_priv = make_private_with_epsilon_safe(
        pe,
        module=policy,
        optimizer=opt,
        data_loader=train_dl,
        epochs=EPOCHS,
        target_epsilon=eps,
        target_delta=DELTA,
        max_grad_norm=MAX_GRAD_NORM,
        poisson_sampling=POISSON,
    )

    KL_PENALTY_NAME = "mean_log_ratio_proxy"

    clip_counts = {k: {"clipped": 0, "total": 0} for k in CLIP_WINDOWS.keys()}
    global_step = 0
    t0 = time.time()
    train_status = "ok"

    try:
        with BatchMemoryManager(
            data_loader=train_dl_priv,
            max_physical_batch_size=PHYSICAL_BATCH_SIZE,
            optimizer=opt
        ) as mem_dl:
            for ep in range(EPOCHS):
                pbar = tqdm(mem_dl, desc=f"[ppo_like cached] train {ep+1}/{EPOCHS}", leave=False)
                for batch in pbar:
                    batch = {k: v.to(DEVICE) for k, v in batch.items() if torch.is_tensor(v)}
                    opt.zero_grad(set_to_none=True)

                    pol_c, nc = seq_logp_response_only_sum_and_count(policy, batch["chosen_input_ids"], batch["chosen_attention_mask"],
                                                                     batch["prompt_len"], tokenizer.pad_token_id)
                    pol_r, nr = seq_logp_response_only_sum_and_count(policy, batch["rejected_input_ids"], batch["rejected_attention_mask"],
                                                                     batch["prompt_len"], tokenizer.pad_token_id)

                    ref_delta = batch["ref_delta"]
                    delta_pol = pol_c - pol_r

                    # ✅ rho stabilization: exp(clamp(log_rho))
                    log_rho = delta_pol - ref_delta
                    log_rho = torch.clamp(log_rho, -C_LOGRHO, C_LOGRHO)
                    rho = torch.exp(log_rho)

                    if not torch.isfinite(rho).all():
                        raise RuntimeError("Non-finite rho detected (after clamp).")

                    rho_clip = torch.clamp(rho, 1.0 - EPS_CLIP, 1.0 + EPS_CLIP)

                    A = batch["adv"]

                    loss_clip = -torch.minimum(rho * A, rho_clip * A)

                    ref_sum_c = batch["ref_sum_c"]
                    ref_sum_r = batch["ref_sum_r"]
                    logratio_c = (pol_c - ref_sum_c) / nc
                    logratio_r = (pol_r - ref_sum_r) / nr
                    kl_proxy = 0.5 * (logratio_c + logratio_r)

                    loss = (loss_clip + BETA_KL * kl_proxy).mean()

                    if not torch.isfinite(loss):
                        raise RuntimeError("Non-finite loss detected in PPO-like.")

                    loss.backward()

                    hits = clip_window_hits(global_step)
                    if hits:
                        norms_sq = per_sample_norms_sq_from_grad_sample(policy)
                        if norms_sq.numel() > 0:
                            norms = torch.sqrt(norms_sq)
                            n = int(norms.numel())
                            n_clipped = int((norms > MAX_GRAD_NORM).sum().item())
                            for w in hits:
                                clip_counts[w]["clipped"] += n_clipped
                                clip_counts[w]["total"] += n

                    opt.step()
                    global_step += 1
                    pbar.set_postfix(loss=float(loss.detach().cpu()), gstep=global_step)

    except Exception as e:
        train_status = f"train_error:{type(e).__name__}"
        print(f"[TRAIN ERROR] {method} | {train_status} | {e}")

    train_minutes = (time.time() - t0) / 60.0
    eps_spent = safe_get_epsilon(pe, DELTA)

    clip_fracs = {k: (clip_counts[k]["clipped"]/clip_counts[k]["total"] if clip_counts[k]["total"]>0 else np.nan)
                  for k in CLIP_WINDOWS.keys()}

    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    row = dict(
        timestamp=ts, run_id=run_id, method=method,
        epsilon_target=float(eps), epsilon_spent=float(eps_spent), delta=float(DELTA),
        seed=int(seed), epochs=int(EPOCHS), lr=float(LR_POL),
        max_grad_norm=float(MAX_GRAD_NORM), poisson_sampling=bool(POISSON),
        n_total=int(N_TOTAL), test_frac=float(TEST_FRAC), max_len=int(MAX_LEN),
        logical_batch=int(LOGICAL_BATCH_SIZE), physical_batch=int(PHYSICAL_BATCH_SIZE),
        metric_name="win_rate", metric_value=np.nan,
        train_minutes=float(train_minutes), eval_minutes=np.nan,
        train_status=train_status, eval_status="train_only",
        artifact_dir=artifact_dir,
        beta=np.nan, beta_kl=float(BETA_KL), eps_clip=float(EPS_CLIP),
        lora_r=8, lora_alpha=32, lora_dropout=0.0, lora_last_k=1,
        split_role="po_half",
        rm_run_id=str(rm_run_id), rm_artifact_dir=str(rm_artifact_dir),
        kl_penalty_name=KL_PENALTY_NAME,
        **clip_fracs
    )
    append_row(MASTER_CSV, row)

    if train_status != "ok":
        update_row(MASTER_CSV, run_id, dict(
            eval_status="skipped_train_error",
            metric_value=np.nan,
            eval_minutes=np.nan,
            epsilon_spent=float(eps_spent),
        ))
        cleanup(policy, opt, pe, train_dl, test_dl, train_dl_priv)
        return

    unwrap_model(policy).save_pretrained(os.path.join(artifact_dir, "adapter"))
    tokenizer.save_pretrained(os.path.join(artifact_dir, "adapter"))

    eval_status = "ok"
    win = np.nan
    t1 = time.time()
    try:
        win = eval_win_rate_policy(policy, test_dl)
    except Exception as e:
        eval_status = f"eval_error:{type(e).__name__}"
        print(f"[EVAL ERROR] {method} | {eval_status} | {e}")
    eval_minutes = (time.time() - t1) / 60.0

    update_row(MASTER_CSV, run_id, dict(
        metric_value=float(win) if win==win else np.nan,
        eval_minutes=float(eval_minutes),
        eval_status=eval_status,
        epsilon_spent=float(safe_get_epsilon(pe, DELTA)),
        train_status="ok",
    ))

    print(f"[DONE] ppo_like_split(cached) eps={eps} seed={seed} | win={(win*100 if win==win else float('nan')):.2f}% | eps_spent={safe_get_epsilon(pe, DELTA):.3f}")
    print("  artifact_dir=", artifact_dir)
    cleanup(policy, opt, pe, train_dl, test_dl, train_dl_priv)

## Load the HH-RLHF data

In [ ]:
raw = load_dataset("Anthropic/hh-rlhf", split=f"train[:{N_TOTAL}]")
print("Loaded raw dataset:", len(raw))

## Run all experiments

The loop trains the reward models and policy baselines for each privacy level and seed, then prints the successful rows in `master_results.csv`.

In [ ]:
for seed in SEEDS:
    print("\n" + "="*110)
    print(f"### PREP DATA for seed={seed} ###")
    print("="*110)

    split = raw.train_test_split(test_size=TEST_FRAC, shuffle=True, seed=seed)
    train_raw = split["train"]
    test_raw  = split["test"]

    try:
        train_tok = train_raw.map(preprocess_pairwise_policy, batched=True, num_proc=4, remove_columns=train_raw.column_names)
        test_tok  = test_raw.map(preprocess_pairwise_policy,  batched=True, num_proc=4, remove_columns=test_raw.column_names)
    except Exception:
        train_tok = train_raw.map(preprocess_pairwise_policy, batched=True, remove_columns=train_raw.column_names)
        test_tok  = test_raw.map(preprocess_pairwise_policy,  batched=True, remove_columns=test_raw.column_names)

    # seed-level ref cache for DPO + PPO
    train_tok = ensure_reference_cache_train(train_tok, seed)

    # deterministic split train_tok -> rm_half / po_half
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(train_tok))
    half = len(idx) // 2
    rm_half = train_tok.select(idx[:half].tolist())
    po_half = train_tok.select(idx[half:].tolist())

    print(f"train_tok={len(train_tok)} | test_tok={len(test_tok)} | rm_half={len(rm_half)} | po_half={len(po_half)}")

    for eps in EPSILONS:
        print("\n" + "-"*110)
        print(f"### RUN eps={eps} seed={seed} ###")
        print("-"*110)

        # A) ours: DP-RM head-only on full train
        run_dp_rm_headonly(eps, seed, train_tok, test_tok, method="dp_rm_postproc", split_role="full")

        # B) DP-DPO (cached ref)
        run_dp_dpo_lastlora_cachedref(eps, seed, train_tok, test_tok)

        # C1) DP-RM head-only on rm_half
        run_dp_rm_headonly(eps, seed, rm_half, test_tok, method="dp_rm_split", split_role="rm_half")

        # locate latest successful dp_rm_split for this eps/seed
        df = load_master_df(MASTER_CSV)
        cand = df[
            (df["method"].astype(str) == "dp_rm_split") &
            (pd.to_numeric(df["epsilon_target"], errors="coerce") == float(eps)) &
            (pd.to_numeric(df["seed"], errors="coerce") == int(seed)) &
            (df["split_role"].astype(str) == "rm_half") &
            (df["train_status"].astype(str) == "ok") &
            (df["eval_status"].astype(str) == "ok")
        ].copy()

        if cand.empty:
            print(f"[WARN] No successful dp_rm_split found for eps={eps}, seed={seed}. Skipping dp_ppo_like_split.")
            continue

        cand["timestamp_dt"] = pd.to_datetime(cand["timestamp"], errors="coerce")
        cand = cand.sort_values(["timestamp_dt", "run_id"], ascending=True)
        rm_row = cand.iloc[-1]
        rm_run_id = str(rm_row["run_id"])
        rm_artifact_dir = str(rm_row["artifact_dir"])

        # cache advantage on po_half using this RM
        po_half_adv = ensure_adv_cache_po(po_half, seed, eps, rm_run_id, rm_artifact_dir)

        # C2) PPO-like split (cached ref + cached adv; rho stabilized)
        run_dp_ppo_like_split_cached(eps, seed, po_half_adv, test_tok, rm_run_id=rm_run_id, rm_artifact_dir=rm_artifact_dir)

    cleanup(train_tok, test_tok, rm_half, po_half, train_raw, test_raw, split)

print("\nALL DONE.")
print("Master CSV:", MASTER_CSV)

df = load_master_df(MASTER_CSV)
succ = df[
    (df["train_status"].astype(str) == "ok") &
    (df["eval_status"].astype(str) == "ok")
][[
    "method","epsilon_target","seed","metric_name","metric_value",
    "lr","max_grad_norm","beta","beta_kl","eps_clip","kl_penalty_name","artifact_dir"
]].copy()

print("\n=== SUCCESSFUL RUNS (train_status==ok AND eval_status==ok) ===")
print(succ.sort_values(["method","epsilon_target","seed"]).to_string(index=False))